Knee concurrency prediction: 8

Target p95 latency (SLO): 3 seconds






In [1]:
import subprocess
import os

# 1. Install python3.10 and venv support on Colab
subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
subprocess.run(["sudo", "apt-get", "install", "python3.10", "python3.10-venv", "python3.10-dev", "-y"], check=True)

# 2. Create the virtual environment
subprocess.run(["python3.10", "-m", "venv", "/content/venv"], check=True)

# 3. Upgrade pip inside the venv
subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"], check=True)

CompletedProcess(args=['/content/venv/bin/python', '-m', 'pip', 'install', '--upgrade', 'pip'], returncode=0)

In [2]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [4]:
import sys
import subprocess
import os

print("Current Python:")
subprocess.run(["python", "--version"])

print("\nPython 3.10:")
subprocess.run(["python3.10", "--version"])

print("\nVenv Python:")
if os.path.exists("/content/venv/bin/python"):
    subprocess.run(["/content/venv/bin/python", "--version"])
else:
    print("Venv does not exist")

Current Python:

Python 3.10:

Venv Python:


In [5]:
import subprocess

# Install Python 3.10
subprocess.run(
    ["sudo", "apt-get", "update", "-y"],
    check=True
)

subprocess.run(
    ["sudo", "apt-get", "install", "-y",
     "python3.10", "python3.10-venv", "python3.10-dev"],
    check=True
)

# Verify
subprocess.run(["python3.10", "--version"], check=True)

# Create venv
subprocess.run(
    ["python3.10", "-m", "venv", "/content/venv"],
    check=True
)

# Upgrade pip inside venv
subprocess.run(
    ["/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"],
    check=True
)

print("Python 3.10 venv is ready!")

Python 3.10 venv is ready!


In [6]:
import subprocess

VENV_PY = "/content/venv/bin/python"

subprocess.run([
    VENV_PY, "-m", "pip", "install", "-q",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)

print("serving pins installed")

serving pins installed


In [7]:
import os
import signal
import subprocess
import sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"
VENV_PY = "/content/venv/bin/python"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args):
    cmd = [VENV_PY, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args=None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 9675, logging to /content/server.log


In [8]:
import time
import urllib.request
import urllib.error

def wait_for_health(port=8000, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print(f"server healthy: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print("TIMED OUT")
    return False

healthy = wait_for_health()

server healthy: http://localhost:8000/v1/models -> 200


In [9]:
%%writefile bench.py

"""Benchmark harness for the serving stack (week 3 day 5 reference).

Sweeps concurrency levels against an OpenAI-compatible endpoint and reports the
numbers the week-3 lab is graded on: tokens/sec, TTFT p50/p95, end-to-end
latency p95, and error counts per level.

CLI contract (the week-3 lab is written against this exactly):

    python bench.py \
        --base-url http://localhost:8000 \
        --model Qwen/Qwen2.5-0.5B-Instruct \
        --concurrency 1,2,4,8,16 \
        --requests-per-level 20 \
        --prompt-file prompts.txt \
        --out bench_report.json

Method:
  - Every request uses stream=true so TTFT is the wall-clock time to the first
    SSE content chunk. End-to-end latency is time to the [DONE] sentinel.
  - One warm-up request per level is fired and excluded from the statistics, so
    a cold cache or JIT does not skew the first measured level.
  - Errors are counted per level and never crash the sweep; a level with all
    errors still reports (with null latencies).
  - Output is written to --out as JSON and printed as a readable table. Re-runs
    append to a `runs` array in the same file when it already exists, so an A/B
    (CPU vs vLLM, Monday vs Wednesday) accumulates in one place.

Dependencies: httpx (async). Pin per ../../PINS.md.
"""

from __future__ import annotations

import argparse
import asyncio
import json
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx


# --------------------------------------------------------------------------- #
# Per-request measurement                                                      #
# --------------------------------------------------------------------------- #

@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None          # time to first content chunk
    latency_s: Optional[float] = None       # time to [DONE]
    completion_tokens: int = 0              # counted from streamed chunks
    error: Optional[str] = None


async def _one_request(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompt: str,
    max_tokens: int,
) -> RequestResult:
    """Fire one streaming completion and measure TTFT and end-to-end latency.

    TTFT is the time from send to the first SSE frame that carries non-empty
    content (the role-announcement frame and empty deltas do not count).
    """
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "temperature": 0.0,
    }
    url = base_url.rstrip("/") + "/v1/chat/completions"
    start = time.perf_counter()
    ttft: Optional[float] = None
    tokens = 0

    try:
        async with client.stream("POST", url, json=body) as response:
            if response.status_code != 200:
                # Drain so the connection can be reused, then report the error.
                text = (await response.aread()).decode("utf-8", "replace")[:200]
                return RequestResult(
                    ok=False, error=f"HTTP {response.status_code}: {text}"
                )
            async for line in response.aiter_lines():
                if not line or not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - start
                    tokens += 1
        latency = time.perf_counter() - start
        return RequestResult(
            ok=True, ttft_s=ttft, latency_s=latency, completion_tokens=tokens
        )
    except Exception as exc:  # network error, timeout, reset: count, do not raise
        return RequestResult(ok=False, error=f"{type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------- #
# Per-level sweep                                                              #
# --------------------------------------------------------------------------- #

@dataclass
class LevelReport:
    concurrency: int
    tokens_per_s: float
    ttft_p50_s: Optional[float]
    ttft_p95_s: Optional[float]
    latency_p95_s: Optional[float]
    errors: int
    ok: int
    wall_s: float = field(default=0.0)


def _percentile(values: list[float], pct: float) -> Optional[float]:
    """Nearest-rank percentile, robust for tiny samples."""
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return round(ordered[0], 4)
    rank = max(1, int(round(pct / 100.0 * len(ordered))))
    rank = min(rank, len(ordered))
    return round(ordered[rank - 1], 4)


async def _run_level(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompts: list[str],
    concurrency: int,
    requests_per_level: int,
    max_tokens: int,
) -> LevelReport:
    """Run one concurrency level: one warm-up (excluded), then the measured batch."""
    # Warm-up, excluded from stats. Ignore its result entirely.
    await _one_request(client, base_url, model, prompts[0], max_tokens)

    # Fire `requests_per_level` requests `concurrency` at a time. A semaphore
    # caps in-flight requests so the level actually holds the intended load.
    semaphore = asyncio.Semaphore(concurrency)

    async def _guarded(index: int) -> RequestResult:
        async with semaphore:
            prompt = prompts[index % len(prompts)]
            return await _one_request(client, base_url, model, prompt, max_tokens)

    level_start = time.perf_counter()
    results = await asyncio.gather(
        *(_guarded(i) for i in range(requests_per_level))
    )
    wall = time.perf_counter() - level_start

    ok = [r for r in results if r.ok]
    errors = len(results) - len(ok)
    ttfts = [r.ttft_s for r in ok if r.ttft_s is not None]
    latencies = [r.latency_s for r in ok if r.latency_s is not None]
    total_tokens = sum(r.completion_tokens for r in ok)

    # System throughput: total completion tokens over the level wall-clock, so
    # it rises with concurrency until the server saturates.
    tokens_per_s = round(total_tokens / wall, 2) if wall > 0 else 0.0

    return LevelReport(
        concurrency=concurrency,
        tokens_per_s=tokens_per_s,
        ttft_p50_s=_percentile(ttfts, 50),
        ttft_p95_s=_percentile(ttfts, 95),
        latency_p95_s=_percentile(latencies, 95),
        errors=errors,
        ok=len(ok),
        wall_s=round(wall, 3),
    )


# --------------------------------------------------------------------------- #
# Orchestration                                                                #
# --------------------------------------------------------------------------- #

def _load_prompts(path: str) -> list[str]:
    with open(path, encoding="utf-8") as handle:
        prompts = [line.strip() for line in handle if line.strip()]
    if not prompts:
        raise SystemExit(f"prompt file {path!r} has no non-empty lines")
    return prompts


def _print_table(levels: list[LevelReport]) -> None:
    header = (
        f"{'conc':>4}  {'tok/s':>8}  {'ttft_p50':>9}  {'ttft_p95':>9}  "
        f"{'lat_p95':>8}  {'ok':>4}  {'err':>4}"
    )
    print(header)
    print("-" * len(header))
    for lv in levels:
        def fmt(value: Optional[float]) -> str:
            return f"{value:.3f}" if value is not None else "  n/a"
        print(
            f"{lv.concurrency:>4}  {lv.tokens_per_s:>8.2f}  "
            f"{fmt(lv.ttft_p50_s):>9}  {fmt(lv.ttft_p95_s):>9}  "
            f"{fmt(lv.latency_p95_s):>8}  {lv.ok:>4}  {lv.errors:>4}"
        )


def _write_report(out_path: str, run_record: dict) -> None:
    """Append this run to `runs` in the JSON file, creating it if absent."""
    document: dict = {"runs": []}
    if os.path.exists(out_path):
        try:
            with open(out_path, encoding="utf-8") as handle:
                existing = json.load(handle)
            if isinstance(existing, dict) and isinstance(existing.get("runs"), list):
                document = existing
        except (json.JSONDecodeError, OSError):
            # Corrupt or unreadable prior file: start fresh rather than crash.
            document = {"runs": []}
    document["runs"].append(run_record)
    with open(out_path, "w", encoding="utf-8") as handle:
        json.dump(document, handle, indent=2)


async def _sweep(args: argparse.Namespace) -> tuple[dict, list[LevelReport]]:
    prompts = _load_prompts(args.prompt_file)
    concurrency_levels = [int(c) for c in args.concurrency.split(",") if c.strip()]

    timeout = httpx.Timeout(args.timeout, connect=10.0)
    limits = httpx.Limits(max_connections=max(concurrency_levels) + 4)
    levels: list[LevelReport] = []

    headers = {}
    if getattr(args, "api_key", ""):
        headers["Authorization"] = "Bearer " + args.api_key
    async with httpx.AsyncClient(timeout=timeout, limits=limits,
                                 headers=headers) as client:
        for concurrency in concurrency_levels:
            report = await _run_level(
                client=client,
                base_url=args.base_url,
                model=args.model,
                prompts=prompts,
                concurrency=concurrency,
                requests_per_level=args.requests_per_level,
                max_tokens=args.max_tokens,
            )
            levels.append(report)
            # Progress line per level so a long sweep is not silent.
            print(
                f"[level {concurrency}] tok/s={report.tokens_per_s} "
                f"ttft_p95={report.ttft_p95_s} errors={report.errors}",
                flush=True,
            )

    return {
        "timestamp": int(time.time()),
        "base_url": args.base_url,
        "model": args.model,
        "requests_per_level": args.requests_per_level,
        "max_tokens": args.max_tokens,
        "prompt_file": args.prompt_file,
        "levels": [vars(lv) for lv in levels],
    }, levels


def main() -> None:
    parser = argparse.ArgumentParser(description="serving-stack benchmark harness")
    parser.add_argument("--base-url", default="http://localhost:8000")
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", default="1,2,4,8,16",
                        help="comma-separated concurrency levels")
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", default="prompts.sample.txt")
    parser.add_argument("--out", default="bench_report.json")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--api-key", default=os.environ.get("API_KEY", ""),
                        help="bearer key for keyed services (or set API_KEY); "
                             "omit for an open endpoint")
    parser.add_argument("--timeout", type=float, default=120.0,
                        help="per-request timeout in seconds")
    args = parser.parse_args()

    run_record, levels = asyncio.run(_sweep(args))
    print()
    _print_table(levels)
    _write_report(args.out, run_record)
    print(f"\nwrote {args.out} (run appended)")


if __name__ == "__main__":
    main()

Writing bench.py


In [10]:
%%writefile prompts.txt
What is a GPU?
Define tokens per second in one line.
Explain the difference between prefill and decode in two sentences.
List three reasons decode is memory-bound rather than compute-bound.
Summarise what an inference server does for an ops team, in three short bullets.
Why does a longer prompt increase time to first token but not the per-token gap?
Describe the KV cache to a new engineer and say why it grows with context length.
Walk through what continuous batching changes versus static batching, with an example of the straggler effect it removes.
Name two things weight-only quantisation trades away in exchange for smaller memory footprint.
A user asks for the weather in Riyadh and the current time in Tokyo; describe the two tool calls you would make and the arguments for each.
Write a short runbook for rolling back a bad deployment, listing the steps in order and the check after each one.
Explain, for a non-technical manager, why a busy GPU is not the same as a productive GPU, using the utilisation trap.
Compare fp16 and int4 for serving a 1.5 billion parameter model: memory, speed, and quality, in a short paragraph each.
Give a one-sentence definition of p95 latency and say why it matters more than the average for an SLO.
Draft three sentences a platform team could send another team to describe an OpenAI-compatible endpoint they can call.
Outline the symptom, hypothesis, and measurement steps you would take when throughput is lower than expected under load.
What is PagedAttention and what problem in KV cache memory does it solve? Answer in two sentences.
Explain why the knee at the SLO, not the peak throughput, is the honest capacity number for a benchmark.
Describe how you would size the GPU memory budget for a model plus its KV cache before ever loading it.
Write a calm status update for a channel of engineers explaining that latency has risen, what you suspect, and what you are doing about it, in four sentences.

Writing prompts.txt


In [11]:
import os

print("bench.py:", os.path.exists("bench.py"))
print("prompts.txt:", os.path.exists("prompts.txt"))

bench.py: True
prompts.txt: True


In [12]:
! /content/venv/bin/python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=48.91 ttft_p95=0.084 errors=0
[level 2] tok/s=102.34 ttft_p95=0.0863 errors=0
[level 4] tok/s=185.21 ttft_p95=0.1022 errors=0
[level 8] tok/s=312.58 ttft_p95=0.1353 errors=0
[level 16] tok/s=522.17 ttft_p95=0.2183 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     48.91      0.061      0.084     2.918    20     0
   2    102.34      0.057      0.086     2.438    20     0
   4    185.21      0.060      0.102     2.603    20     0
   8    312.58      0.131      0.135     3.060    20     0
  16    522.17      0.213      0.218     3.008    20     0

wrote bench_report.json (run appended)


In [13]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )

TARGET_P95_S = 3.0

under = [
    L for L in levels
    if L["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under,
    key=lambda L: L["concurrency"]
) if under else None

print("knee:", knee)

c= 1  tok/s=   48.9  ttft_p95=0.084  lat_p95=2.918  errors=0
c= 2  tok/s=  102.3  ttft_p95=0.086  lat_p95=2.438  errors=0
c= 4  tok/s=  185.2  ttft_p95=0.102  lat_p95=2.603  errors=0
c= 8  tok/s=  312.6  ttft_p95=0.135  lat_p95=3.060  errors=0
c=16  tok/s=  522.2  ttft_p95=0.218  lat_p95=3.008  errors=0
knee: {'concurrency': 4, 'tokens_per_s': 185.21, 'ttft_p50_s': 0.0602, 'ttft_p95_s': 0.1022, 'latency_p95_s': 2.6033, 'errors': 0, 'ok': 20, 'wall_s': 11.706}


In [14]:
import json

knee = {
    "concurrency": 4,
    "tokens_per_s": 185.21,
    "wall_s": 11.706
}

request_rate = knee["concurrency"] / knee["wall_s"]

note = f"""# capacity-note

## Capacity note (team, one page)

### The numbers

Locked model: Qwen/Qwen2.5-1.5B-Instruct

Target p95 end-to-end latency (your SLO today): 3.0 seconds

Knee concurrency (highest concurrency whose p95 is still under target): 4

Tokens per second at the knee: 185.21

Max sustainable request rate at the target p95: {request_rate:.3f} req/s

### The limiting family

Compute-bound: throughput continues increasing as concurrency rises, but p95 latency crosses the 3-second SLO at higher concurrency.

### Why the knee, not the peak

We report the knee because it is the highest concurrency that meets the latency SLO, while the peak throughput at higher concurrency violates the required response-time target.
"""

with open("capacity-note.md", "w") as f:
    f.write(note)

with open("knee.json", "w") as f:
    json.dump({
        "target_p95_s": 3.0,
        "knee_concurrency": 4
    }, f, indent=2)

print("capacity-note.md and knee.json created.")
print(f"Max sustainable request rate: {request_rate:.3f} req/s")

capacity-note.md and knee.json created.
Max sustainable request rate: 0.342 req/s


In [15]:
import json

report = json.load(open("bench_report.json"))
levels = report["runs"][-1]["levels"]

knee = max(
    [L for L in levels if L["latency_p95_s"] <= 3.0],
    key=lambda L: L["concurrency"]
)

print(f"Knee concurrency: {knee['concurrency']}")
print(f"Tokens/s at knee: {knee['tokens_per_s']:.2f}")
print(f"p95 latency at knee: {knee['latency_p95_s']:.3f} s")
print(f"Max sustainable request rate: {knee['concurrency']/knee['wall_s']:.3f} req/s")

# Clean shutdown
server.terminate()
print("Server stopped.")

Knee concurrency: 4
Tokens/s at knee: 185.21
p95 latency at knee: 2.603 s
Max sustainable request rate: 0.342 req/s
Server stopped.


In [16]:

%%writefile verify_cell.py

# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


Writing verify_cell.py


In [17]:
!python verify_cell.py

levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [18]:
import os

for f in ["bench_report.json", "capacity-note.md", "knee.json", "verify_cell.py"]:
    print(f, os.path.exists(f))

bench_report.json True
capacity-note.md True
knee.json True
verify_cell.py True


In [21]:
!git clone https://github.com/layankhalid0/week3.git /content/week3

Cloning into '/content/week3'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 34 (delta 7), reused 27 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 27.34 KiB | 500.00 KiB/s, done.
Resolving deltas: 100% (7/7), done.


In [22]:
%cd /content/week3
!git checkout W3D5

/content/week3
error: pathspec 'W3D5' did not match any file(s) known to git


In [23]:
!git branch -a

* W3D1
  remotes/origin/HEAD -> origin/W3D1
  remotes/origin/W3D1
  remotes/origin/W3D1-extra-memory-leak
  remotes/origin/W3D2
  remotes/origin/W3D2-extra-paged-kv
  remotes/origin/W3D3
  remotes/origin/W3D4
  remotes/origin/W3D4-extra-quantisation-drift


In [24]:
!git checkout -b W3D5 origin/W3D4

Branch 'W3D5' set up to track remote branch 'W3D4' from 'origin'.
Switched to a new branch 'W3D5'
